In [2]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
# 이미지, 영상, 텍스트, 음성  -> vector space 
# 고차원 데이터의 저차원 압축 표현
# (3, 1024, 1024 ) -> (32 , 32)
# (100, 768, 50000) -> ~~~~~

In [5]:
llm = ChatOpenAI(model='gpt-4o-mini')
embeddings_model = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [7]:
test_emb = embeddings_model.embed_query("Hello")

In [8]:
len(test_emb)

1536

In [9]:
test_emb[:10]

[0.019195556640625,
 -0.0645751953125,
 -0.0016889572143554688,
 0.078125,
 0.0216827392578125,
 -0.0155487060546875,
 -0.015045166015625,
 0.0457763671875,
 -0.005886077880859375,
 -0.045257568359375]

In [10]:
import tiktoken

In [12]:
# !pip install tiktoken

In [13]:
enc = tiktoken.encoding_for_model('gpt-4o-mini')  # embedding space, representation space, manifold

In [14]:
text = "안녕하세요, 오늘 LLM에 대해 배워보겠습니다"
tokens = enc.encode(text)

In [15]:
tokens

[14307,
 171731,
 11,
 106820,
 451,
 19641,
 3107,
 67946,
 33628,
 33771,
 8122,
 105216]

In [16]:
enc.decode(tokens)

'안녕하세요, 오늘 LLM에 대해 배워보겠습니다'

In [19]:
for i, token_id in enumerate(tokens):
    token_text = enc.decode([token_id])
    print(f" token {i+1} : ID : {token_id} -> {token_text}")

 token 1 : ID : 14307 -> 안
 token 2 : ID : 171731 -> 녕하세요
 token 3 : ID : 11 -> ,
 token 4 : ID : 106820 ->  오늘
 token 5 : ID : 451 ->  L
 token 6 : ID : 19641 -> LM
 token 7 : ID : 3107 -> 에
 token 8 : ID : 67946 ->  대해
 token 9 : ID : 33628 ->  배
 token 10 : ID : 33771 -> 워
 token 11 : ID : 8122 -> 보
 token 12 : ID : 105216 -> 겠습니다


In [ ]:
# 1M : input $ 0.05  output $ 0.2

In [23]:
long_text = """인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.

최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한  변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.

LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.

RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다."""


In [ ]:
# chunk
# Splitter
# CharacterTextSplitter
# RecursiveCharacterSplitter
# from_tiktoken_encoder()

In [22]:
# !pip install langchin-text-splitters

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [24]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, separators=["\n\n", "\n", ". ", " ", ""])

In [25]:
chunks = splitter.split_text(long_text)

In [26]:
len(chunks)

7

In [29]:
for i, chunk in enumerate(chunks):
    print(f"[chunk {i+1}] : {len(chunk)} characters")
    print(chunk)

[chunk 1] : 65 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 47 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한  변화를 겪고 있습니다
[chunk 3] : 65 characters
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 46 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
[chunk 5] : 94 characters
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 40 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
[chunk 7] : 61 characters
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [30]:
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, separators=["\n\n", "\n", ". ", " ", ""])

In [32]:
chunks = splitter.split_text(long_text)
for i, chunk in enumerate(chunks):
    print(f"[chunk {i+1}] : {len(chunk)} characters")
    print(chunk)

[chunk 1] : 46 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을
[chunk 2] : 24 characters
지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 3] : 47 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한  변화를 겪고 있습니다
[chunk 4] : 50 characters
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인
[chunk 5] : 24 characters
분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 6] : 46 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
[chunk 7] : 40 characters
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며,
[chunk 8] : 49 characters
기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템
[chunk 9] : 17 characters
시스템 구축에 특히 유용합니다.
[chunk 10] : 40 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
[chunk 11] : 49 characters
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여
[chunk 12] : 20 characters
청크를 검색하여 LLM에 전달합니다.


In [33]:
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                        model_name = 'gpt-4o-mini', chunk_size=50, chunk_overlap=10)

In [35]:
def count_token(text, model = 'gpt-4o-mini'):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [37]:
chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
    tokens = count_token(chunk)
    print(f"[chunk {i+1}] : {tokens} tokens")
    print(chunk)

[chunk 1] : 40 tokens
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 48 tokens
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한  변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를
[chunk 3] : 15 tokens
처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 47 tokens
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을
[chunk 5] : 27 tokens
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 49 tokens
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를
[chunk 7] : 17 tokens
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [38]:
test_text = """서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다.

서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다.

교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다."""

In [ ]:
# 한국어 test_text를 chunking 해보세요. 다양한 chunk size, overlap size를 이용

In [39]:
def split_and_report(text, chunk_size=50, chunk_overlap=10):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                        model_name = 'gpt-4o-mini', chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_text(text)
    token_counts = [count_token(c) for c in chunks]
    
    print(f" chunking report (chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap})")
    for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
        first_line = chunk.split('\n')[0][:40]
        print(f" [{i+1}] {tc} tokens, {len(chunk)} characters | {first_line}")
    
    print(f"summary : {len(chunks)} chunks")
    print(f"avg : {np.mean(token_counts)}")
    print(f"max : {max(token_counts)}")

In [40]:
split_and_report(test_text)

 chunking report (chunk_size = 50, chunk_overlap = 10)
 [1] 43 tokens, 73 characters | 서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북
 [2] 43 tokens, 57 characters | 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등
 [3] 47 tokens, 79 characters | 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다.
summary : 3 chunks
avg : 44.333333333333336
max : 47


In [41]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [42]:
embedding_model = OpenAIEmbeddings(model = 'text-embedding-3-small')
# embedding_model('hello')

In [43]:
query = "인공지능이 세상을 바꾸고 있습니다"
query_vector = embedding_model.embed_query(query)

In [45]:
len(query_vector)

1536

In [57]:
query_vector[:5]

[0.0316162109375,
 0.028289794921875,
 0.0098724365234375,
 0.033111572265625,
 -0.0077667236328125]

In [ ]:
# 15000 : 주어와 목적어사이의 관계, 주어랑 다음에 올지 모르는 단어와 동사의 관계 ..
# 1500 : 주어, 목적어, 동사, 그들의 관계, 

In [46]:
texts = [
    "AI가 세상을 바꾸고 있습니다",
    "AI가 의료분야를 혁신하고 있다",
    "머신러닝으로 질병을 예측할 수 있다",
    "오늘 저녁에 치킨을 시켜먹었다"
]

doc_vectors = embedding_model.embed_documents(texts)

In [48]:
len(doc_vectors)

4

In [49]:
for vec in doc_vectors:
    print(len(vec))

1536
1536
1536
1536


In [51]:
doc_vectors[0][:5]

[0.0274810791015625,
 0.016571044921875,
 -0.00400543212890625,
 0.03912353515625,
 -0.008758544921875]

In [52]:
doc_vectors[1][:5]

[-0.041717529296875,
 -0.0005006790161132812,
 0.01273345947265625,
 0.06964111328125,
 0.001049041748046875]

In [53]:
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
# !pip install scikit-learn

In [56]:
cosine_similarity(doc_vectors)

array([[1.        , 0.45889463, 0.15164974, 0.12933841],
       [0.45889463, 1.        , 0.26785045, 0.15696454],
       [0.15164974, 0.26785045, 1.        , 0.10156876],
       [0.12933841, 0.15696454, 0.10156876, 1.        ]])

In [ ]:
texts = [
    "AI가 세상을 바꾸고 있습니다",
    "AI가 의료분야를 혁신하고 있다",
    "머신러닝으로 질병을 예측할 수 있다",
    "오늘 저녁에 치킨을 시켜먹었다"
]

In [58]:
all_vectors = [query_vector] + doc_vectors
all_texts = [query] + texts

cosine_similarity(all_vectors)

array([[1.        , 0.77057633, 0.3211806 , 0.21654351, 0.05781237],
       [0.77057633, 1.        , 0.45889463, 0.15164974, 0.12933841],
       [0.3211806 , 0.45889463, 1.        , 0.26785045, 0.15696454],
       [0.21654351, 0.15164974, 0.26785045, 1.        , 0.10156876],
       [0.05781237, 0.12933841, 0.15696454, 0.10156876, 1.        ]])

In [59]:
similarity_matrix = cosine_similarity(all_vectors)
labels = ["query"] + [f"doc{i+1}" for i in range(len(texts))]

df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(df)

          query      doc1      doc2      doc3      doc4
query  1.000000  0.770576  0.321181  0.216544  0.057812
doc1   0.770576  1.000000  0.458895  0.151650  0.129338
doc2   0.321181  0.458895  1.000000  0.267850  0.156965
doc3   0.216544  0.151650  0.267850  1.000000  0.101569
doc4   0.057812  0.129338  0.156965  0.101569  1.000000


In [69]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}


def classify(text, categories):
    cat_vectors = {}
    for cat, examples in categories.items():
        embs = embedding_model.embed_documents(examples)
        cat_vectors[cat] = np.mean(embs, axis=0)
        
    text_emb = embedding_model.embed_query(text)
    
    scores = {}
    for cat, cat_vec in cat_vectors.items():
        sim = cosine_similarity([text_emb], [cat_vec])[0][0]
        scores[cat] = round(sim, 4)
    
    best_cat = max(scores, key=scores.get)
    
    print(f"입력 : {text}")
    print(f"예측 : {best_cat}")
    for c, s in scores.items():
        marker = " <<<" if c == best_cat else ""
        print(f" {c}: {s}{marker}")
              
    return best_cat, scores

In [70]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}


test_senteces = ["새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다", "올해 올림픽에서 한국이 좋은 성적을 거뒀다", "이 식당 불고기가 정말 맛있다"]

for sent in test_senteces:
    print()
    classify(sent, categories)


입력 : 새로운 GPU가 출시되어 AI 학습속도가 빨라졌습니다
예측 : 기술
 기술: 0.3836 <<<
 스포츠: 0.2215
 음식: 0.1105

입력 : 올해 올림픽에서 한국이 좋은 성적을 거뒀다
예측 : 스포츠
 기술: 0.1582
 스포츠: 0.5186 <<<
 음식: 0.3005

입력 : 이 식당 불고기가 정말 맛있다
예측 : 음식
 기술: 0.134
 스포츠: 0.1711
 음식: 0.4692 <<<


In [ ]:
# CacheBackedEmbeddings

In [71]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [74]:
store = InMemoryByteStore()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
                    embedding_model, store, namespace='embedding-cache')